# 02 · Preprocessing — Domestic Flight Price (Regression)

**Goal:** turn the raw data into a clean numeric matrix that every model in Phase 3 can use, and set up one cross-validation routine that scores every model the same way. Every step here is justified by a finding in `01_eda.ipynb`.

The reusable code lives in `src/` so all models share it; this notebook calls it step by step and **checks that each step did what we claimed**.

| Step | What | Where it runs | EDA reason |
|---|---|---|---|
| 1 | Fill missing `airline` from the flight-code prefix | once, before CV | 2.7: prefix → airline is exact |
| 2 | Drop exact duplicate rows | once, before CV | 2.7: copies leak across folds |
| 3 | Fill missing `duration` from same flight + route | **inside** CV | 2.7: ~98% recoverable |
| 4 | Encode features (scale, ordinal, one-hot) | **inside** CV | 2.3–2.6 |
| 5 | CV harness + baselines | — | gives every later model a reference score |
| 6 | Baseline submission | — | tests the full pipeline end to end on Kaggle |

**Key idea — "once before CV" vs "inside CV":** a step can run once on all the data only if it learns nothing from it (a fixed lookup rule). A step that *learns* numbers from data (a median, a mean for scaling, the list of categories) must be re-learned inside each CV fold from the training part only. Otherwise the validation rows have quietly influenced the model, and the CV score is optimistic. This is called **data leakage**.

**What this cell does:** Standard opening: auto-reload, make `src/` importable, import `config`.

**Why:** Same as every notebook; lets us import our shared preprocessing and evaluation code, locally or on Kaggle.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

# Make `src` importable: repo root locally, or an uploaded Kaggle dataset containing src/
for p in [Path.cwd(), *Path.cwd().parents, *(f.parent.parent for f in Path("/kaggle/input").glob("**/src/config.py"))]:
    if (p / "src" / "config.py").exists():
        sys.path.insert(0, str(p.resolve()))
        break

from src import config

**What this cell does:** Imports libraries and our project modules (`preprocess`, `models`, `evaluate`), then loads the **raw** data.

**Why:** We load raw data here (not the cleaned version) so the notebook can show the before/after of each step.

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

from src import evaluate
from src.regression import preprocess as pp
from src.regression.models import GroupMedianRegressor

TASK = "regression"
pd.set_option("display.width", 140)

train_raw, test_raw, sample_sub = config.load(TASK)
print("raw train", train_raw.shape, "| raw test", test_raw.shape)

raw train (15589, 12) | raw test (3898, 11)


## Step 1 — Recover `airline` from the flight code

EDA 2.7 showed that every flight-code prefix (`6E` in `6E 285`) belongs to exactly one airline. So a missing airline isn't unknown at all — it's written in the flight code. This is a **fixed lookup**, it never looks at prices, so it's safe to run once on train and test together.

**What this cell does:** Builds the prefix → airline table from all rows (train + test) where the airline is known, and prints it.

**Why:** `build_prefix_map` also raises an error if any prefix maps to two airlines — so if the data ever broke the 1-to-1 assumption, we'd find out immediately instead of silently filling wrong values.

In [3]:
prefix_map = pp.build_prefix_map(train_raw, test_raw)
print(len(prefix_map), "prefixes")
prefix_map.sort_index().to_frame("airline").T

20 prefixes


prefix,6E,9I,AI,BG,EK,EY,FZ,GF,IX,KU,MH,QP,QR,RA,S5,SG,SQ,TG,UL,WY
airline,Oceanic Airlines,Massive Dynamic,Wayfarer Aviation,Tyrell Aeronautics,Stark Aviation,Nakatomi Airlines,Initech Air,Aperture Aviation,Wayfarer Aviation,Black Mesa Air,Virtucon,Sandpiper Air,Vault-Tec Air,Cyberdyne Flights,Hooli Airways,Vandelay Aviation,Weyland Air,Wayne Enterprises Air,Ajira Airways,Omni Consumer Flights


**What this cell does:** Applies the lookup to train and test and compares missing-airline counts before and after.

**Why:** Checks the claim from EDA: every missing airline should be filled (0 left).

In [4]:
train_air = pp.recover_airline(train_raw, prefix_map)
test_air = pp.recover_airline(test_raw, prefix_map)

pd.DataFrame({"before": [train_raw.airline.isna().sum(), test_raw.airline.isna().sum()],
              "after": [train_air.airline.isna().sum(), test_air.airline.isna().sum()]},
             index=["train", "test"])

,before,after
train,3089,0
test,799,0


**Observations (Step 1)**

- 20 prefixes, each mapping to exactly one airline (no conflicts, so the function didn't raise). Wayfarer Aviation is the only airline with two prefixes (AI, IX).
- **All 3,089 train and 799 test missing airlines were filled; 0 remain.** The test-only airline `Tyrell Aeronautics` gets its own prefix (BG).
- For the report: this isn't an estimate; the airline is fully determined by the flight code. It adds zero error and uses no price information, so it's safe to apply before CV.

## Step 2 — Drop exact duplicate rows

If the same row appears twice and CV puts one copy in the training part and the other in the validation part, the model is "validated" on a row it has already memorised. Dropping copies keeps the CV score honest. We keep one copy, so no information is lost.

**What this cell does:** Counts exact duplicates (all features + price identical) **before** and **after** airline recovery, then drops them from the recovered data.

**Why:** Order matters: a row with a missing airline and its twin with the airline filled in look different until Step 1 fills the gap — then they turn out to be identical. So we must recover first, then de-duplicate.

In [5]:
cols = [c for c in train_raw.columns if c != pp.ID]
dups_before = train_raw.duplicated(subset=cols).sum()
dups_after_recovery = train_air.duplicated(subset=cols).sum()
print(f"exact duplicates in raw data:              {dups_before}")
print(f"exact duplicates after airline recovery:   {dups_after_recovery}")

train_clean = pp.drop_exact_duplicates(train_air)
print(f"train rows: {len(train_raw)} -> {len(train_clean)}")

exact duplicates in raw data:              279
exact duplicates after airline recovery:   422
train rows: 15589 -> 15167


**What this cell does:** Uses `load_clean()`, the one-call version of Steps 1–2 that later notebooks will use, and checks that it gives the same result as doing the steps by hand above.

**Why:** Phase 3 notebooks will just call `load_clean()`; this confirms that function does exactly what we demonstrated. It also splits features (X) from the target (y) and drops `id`.

In [6]:
X, y, X_test, test_ids, sample_sub = pp.load_clean()
assert len(X) == len(train_clean) and X.airline.isna().sum() == 0
print("X", X.shape, "| y", y.shape, "| X_test", X_test.shape)
print("feature columns:", list(X.columns))

X (15167, 10) | y (15167,) | X_test (3898, 10)
feature columns: ['airline', 'flight', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'flight_class', 'duration', 'days_left']


**Observations (Step 2)**

- Raw data has 279 exact duplicates, but **after airline recovery there are 422**. 143 more pairs were hidden only because one copy had a missing airline.
- Train goes from 15,589 → **15,167 rows** (2.7% removed).
- This is a nice detail for the viva: the order of preprocessing steps matters. De-duplicating before recovering would have left 143 leaking pairs in the data.
- `load_clean()` reproduces the manual steps exactly, so later notebooks can call it with confidence.

## Step 3 — Fill missing `duration`

The same scheduled flight on the same route with the same stops takes (almost) the same time. `DurationImputer` fills a missing duration with, in order:
1. the median duration of the same **(flight, source, destination, stops)** group,
2. if that group has no known duration: the median of the same **(source, destination, stops)**,
3. if even that is unknown: the overall median.

Unlike Step 1, this **learns medians from data**, so it goes inside the model pipeline and is re-fitted in every CV fold.

**What this cell does:** Fits the imputer on the training data and reports how every missing duration in train and test would be filled (which of the three rules is used).

**Why:** Shows how much of the imputation is a near-exact group lookup vs a rougher fallback — the more that comes from rule 1, the less noise imputation adds.

In [7]:
imp = pp.DurationImputer().fit(X)
pd.DataFrame({"train": imp.fill_report(X), "test": imp.fill_report(X_test)})

,train,test
missing,3065,776
group median,2961,752
route+stops median,104,24
global median,0,0


**What this cell does:** **Tests the imputer on values we actually know.** Hides 15% of the known durations, fits the imputer on the rest, fills the hidden ones, and compares the guesses with the true values.

**Why:** We can't check imputed values against the truth for rows that are really missing. Hiding values we *do* know simulates missingness, so we can measure the imputer's real error — a number worth quoting in the report instead of just asserting “the fill is good”.

In [8]:
rng = np.random.default_rng(config.SEED)
known_idx = X.index[X.duration.notna()]
hide = rng.choice(known_idx, size=int(0.15 * len(known_idx)), replace=False)

X_masked = X.copy()
X_masked.loc[hide, "duration"] = np.nan
filled = pp.DurationImputer().fit(X_masked).transform(X_masked).loc[hide, "duration"]
truth = X.loc[hide, "duration"]

err = (filled - truth).abs()
print(f"hidden values: {len(hide)}")
print(f"exactly right (within 0.01 h): {(err < 0.01).mean():.1%}")
print(f"mean absolute error: {err.mean():.2f} h | median: {err.median():.2f} h | 95th pct: {err.quantile(0.95):.2f} h")
print(f"for comparison, filling with one overall median would give MAE {(truth - X_masked.duration.median()).abs().mean():.2f} h")

hidden values: 1815
exactly right (within 0.01 h): 86.3%
mean absolute error: 0.34 h | median: 0.00 h | 95th pct: 1.00 h
for comparison, filling with one overall median would give MAE 2.25 h


**Observations (Step 3)**

- On the cleaned train set, 3,065 durations are missing. **2,961 (96.6%) are filled from the same flight-route-stops group**, 104 from the route+stops median, and 0 need the global median. Test: 752 of 776 (96.9%) from the group.
- **Masking test:** of 1,815 known durations we hid on purpose, the imputer got **86.3% exactly right**, with mean error **0.34 h** (median 0 h). Filling with one overall median would have given **2.25 h** mean error, about 6.6× worse.
- The imputer is fitted per CV fold, so each fold's validation rows never contribute to the medians.

## Step 4 — Encode features into numbers

Linear models and sklearn trees need a purely numeric matrix. Each column is encoded according to what the EDA showed about it:

| Column | Encoding | Why |
|---|---|---|
| `duration`, `days_left` | standardised (mean 0, std 1) | puts them on the same scale as the 0/1 columns; required for Ridge/Lasso to penalise coefficients fairly |
| `stops` | ordinal 0 / 1 / 2 (then standardised) | it has a natural order (2.4) |
| `airline` | one-hot; airlines with < 50 training rows share one **"infrequent"** column | 9 airlines have tiny counts (2.3) — their own coefficients would be noise; this bucket also catches the test-only airline |
| cities, time slots, class | one-hot; unseen values ignored | no natural order; few categories |
| `id`, raw `flight` code | dropped | `id` is a row number; ~1,070 flight codes are too many to one-hot |

Like Step 3, encoders learn things (means, the category list), so they also live inside the pipeline.

**What this cell does:** Builds the full preprocessing pipeline (Step 3 imputer + Step 4 encoders), fits it on the training data, and transforms both train and test.

**Why:** This is the exact object every Phase 3 model will start with. Checking the output shape and column names now catches encoding mistakes before they hide inside model scores.

In [9]:
pre = pp.make_preprocessor()
display(pre)

M = pre.fit_transform(X)
M_test = pre.transform(X_test)
print("train matrix", M.shape, "| test matrix", M_test.shape)
print("missing values left — train:", int(M.isna().sum().sum()), "| test:", int(M_test.isna().sum().sum()))
assert list(M.columns) == list(M_test.columns)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('duration', ...), ('columns', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('stops', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature

train matrix (15167, 40) | test matrix (3898, 40)
missing values left — train: 0 | test: 0


**What this cell does:** Lists the 40 output columns grouped by source, and shows which airlines were put into the shared “infrequent” column.

**Why:** Makes the encoding auditable: we can see exactly which features a model receives, and confirm the rare airlines (and the test-only airline) end up in the bucket rather than getting their own unreliable column.

In [10]:
groups = pd.Series(M.columns).str.extract(r"^(airline|source_city|destination_city|departure_time|arrival_time|flight_class)")[0]
print(groups.fillna("numeric / ordinal").value_counts().to_string())

enc = pre.named_steps["columns"].named_transformers_["airline"]
print("\nairlines in the 'infrequent' column:", list(enc.infrequent_categories_[0]))
new_in_test = sorted(set(X_test.airline) - set(X.airline))
print("test-only airlines:", new_in_test, "-> infrequent column =",
      M_test.loc[X_test.airline.isin(new_in_test).to_numpy(), "airline_infrequent_sklearn"].tolist())

0
airline              11
source_city           6
destination_city      6
arrival_time          6
departure_time        6
numeric / ordinal     3
flight_class          2

airlines in the 'infrequent' column: ['Aperture Aviation', 'Black Mesa Air', 'Cyberdyne Flights', 'Hooli Airways', 'Initech Air', 'Massive Dynamic', 'Omni Consumer Flights', 'Weyland Air']
test-only airlines: ['Tyrell Aeronautics'] -> infrequent column = [1.0]


**What this cell does:** Shows the numeric columns' mean and standard deviation after scaling, on train and on test.

**Why:** Sanity check for scaling: on train they should be ~0 and ~1 by construction. On test they should be close but not exact — the scaler used *train's* mean and std, which is exactly what we want (test must be transformed with numbers learned from train only).

In [11]:
num_cols = ["duration", "days_left", "stops"]
pd.DataFrame({"train mean": M[num_cols].mean(), "train std": M[num_cols].std(),
              "test mean": M_test[num_cols].mean(), "test std": M_test[num_cols].std()}).round(3)

,train mean,train std,test mean,test std
duration,-0.0,1.0,-0.010,0.982
days_left,-0.0,1.0,-0.005,1.014
stops,-0.0,1.0,-0.007,0.994


**Observations (Step 4)**

- **40 numeric columns, no missing values**, same columns for train and test.
  - 3 numeric/ordinal (duration, days_left, stops)
  - 11 airline (10 airlines + 1 "infrequent")
  - 6 + 6 cities
  - 6 + 6 time slots
  - 2 class
- 8 airlines share the infrequent column (each has < 50 rows after recovery; Vault-Tec has enough rows after recovery to keep its own column). The test-only `Tyrell Aeronautics` correctly lands in the infrequent column instead of crashing the encoder.
- Scaled columns are exactly mean 0 / std 1 on train and close to that on test (|mean| ≤ 0.01, std 0.98–1.01). That's expected and correct: test is transformed with train's statistics.
- Note: the one-hot encoding keeps *every* category (no column dropped), so for plain OLS the columns are linearly dependent (the "dummy variable trap"). sklearn's solver still finds a solution, but individual coefficients can't be interpreted. In Phase 3, plain OLS should use `drop="first"`. Ridge/Lasso don't need it because the penalty resolves the ambiguity.

## Step 5 — Cross-validation harness and baselines

`evaluate.cross_validate()` is the single scoring routine for **every** model in this project:
- **5-fold KFold, shuffled with `SEED = 42`** → every model is tested on exactly the same 5 splits, so score differences come from the models, not from luck of the split.
- The whole model (preprocessing included) is **cloned and re-fitted per fold** → no leakage.
- Reports **RMSE, MAE, R², RMSLE**, because Kaggle hasn't told us which metric it uses.
- Appends a row to `results/regression/experiments.csv` → that file becomes the report's model table.

**Baselines** are deliberately dumb models. They answer: *how good is "good"?* A real model that can't clearly beat "predict the median price of the ticket's class" has learned nothing useful.

**What this cell does:** Runs CV for two baselines — (a) predict the overall median price for every ticket, (b) predict the median price of the ticket's class — and shows mean ± std across the 5 folds.

**Why:** (a) is the absolute floor. (b) uses only the single strongest feature from EDA (class, R² 0.78 on log price), so it shows how much one feature already gets us. Every Phase 3 model must beat (b).

In [12]:
baselines = {
    "baseline_global_median": GroupMedianRegressor(),
    "baseline_class_median": GroupMedianRegressor(group_col="flight_class"),
}
results = {}
for name, model in baselines.items():
    folds = evaluate.cross_validate(model, X, y, TASK, name, params=model.get_params(),
                                    notes="Phase 2 baseline")
    results[name] = evaluate.summarize(folds)
pd.DataFrame(results).T

,rmse,mae,r2,rmsle
baseline_global_median,43962.7057 ± 1037.3956,22321.1036 ± 783.7697,-0.2502 ± 0.0162,1.1177 ± 0.0225
baseline_class_median,33151.5727 ± 854.8586,13397.1486 ± 475.4353,0.2891 ± 0.0097,0.4978 ± 0.0083


**What this cell does:** Shows the experiment log as it stands.

**Why:** Confirms that both runs were recorded; this is the table that grows through Phase 3 and ends up in the report.

In [13]:
pd.read_csv(config.results_dir(TASK) / "experiments.csv")

,timestamp,model,params,n_rows,rmse_mean,rmse_std,mae_mean,mae_std,r2_mean,r2_std,rmsle_mean,rmsle_std,notes,public_lb
0,2026-10-11T19:05:50,baseline_global_median,"{""group_col"": null}",15167,43962.705715,1037.395585,22321.103584,783.769699,-0.250229,0.01618,1.117705,0.022453,Phase 2 baseline,NaN
1,2026-10-11T19:05:50,baseline_class_median,"{""group_col"": ""flight_class""}",15167,33151.572661,854.858635,13397.148553,475.435308,0.289123,0.00971,0.497770,0.008317,Phase 2 baseline,NaN


**Observations (Step 5)**

- **Global median:** RMSE ₹43,963, R² **−0.25**. Negative R² means it's worse than predicting the *mean*. That's expected: on skewed data the median sits far below the mean, so it underestimates every Business ticket.
- **Class median:** RMSE ₹33,152, MAE ₹13,397, R² **0.29**, RMSLE 0.50.
- Fold-to-fold std is small (~2–3% of the mean), so differences between models larger than that are real, not split noise.
- **Teaching point:** in the EDA, class explained R² 0.78 of *log* price, yet here the class median explains only 0.29 of *raw* price. Same feature, very different numbers, because on the raw scale a handful of very expensive Business tickets (₹100k–₹320k) dominate the squared error, and one median per class can't reach them. This is exactly why the metric (raw vs log scale) changes which model looks best.
- Both runs are logged in `results/regression/experiments.csv`.

## Step 6 — Baseline submission

Before spending effort on real models, we send the class-median baseline to Kaggle. This tests the **whole path** — prediction, file format, column names, row order — while the stakes are zero.

There's a bonus: Kaggle hasn't told us its metric, but we know what each metric *should* read for this baseline (Step 5 table). The score Kaggle returns tells us which one it is.

**What this cell does:** Fits the class-median baseline on all training rows, predicts the test set, checks the file against `sample_submission.csv` (shape, id order, no missing values), and saves it to `submissions/regression/`.

**Why:** `make_submission` refuses to write a file with the wrong shape or id order — a broken submission wastes one of our limited daily Kaggle attempts.

In [14]:
path, sub = evaluate.make_submission(baselines["baseline_class_median"], X, y, X_test, test_ids,
                                     sample_sub, TASK, "baseline_class_median")
print("saved:", path.relative_to(config.ROOT) if not config.ON_KAGGLE else path)
print("shape:", sub.shape, "| columns:", list(sub.columns))
print(sub[sub.columns[1]].value_counts().rename("rows").to_string())
sub.head()

saved: submissions\regression\2026-10-11_baseline_class_median.csv
shape: (3898, 2) | columns: ['id', 'price']
price
7580.0     2337
38546.0    1561


,id,price
0,1,7580.0
1,2,7580.0
2,3,38546.0
3,4,38546.0
4,5,7580.0


**What this cell does:** Lists the score this baseline got in CV under each candidate metric — the lookup table for identifying Kaggle's metric.

**Why:** When Kaggle shows the public score for this file, match it against this table: the metric whose CV value is closest is almost certainly the one Kaggle uses. (Values won't match exactly — different rows — but the four metrics are on wildly different scales.)

In [15]:
guide = evaluate.cross_validate(baselines["baseline_class_median"], X, y, TASK, "", log=False).mean()
pd.DataFrame({"expected Kaggle score if metric is…": guide.round(4)})

,expected Kaggle score if metric is…
rmse,33151.5727
mae,13397.1486
r2,0.2891
rmsle,0.4978


**Observations (Step 6)**

- Submission saved as `submissions/regression/2026-10-11_baseline_class_median.csv`: **3,898 × 2, columns `id, price`**, ids in sample order, no missing values. It only contains two distinct values (₹7,580 Economy, ₹38,546 Business), as a class-median baseline should.
- **Kaggle result (11 Oct): public score 0.31495.** Our CV predicted R² 0.289, and the leaderboard ranks higher scores better (top teams ≈ 0.96). With RMSLE, lower would be better and the value would be near 0.50. **So the metric is R² on raw price.**
- Public 0.315 vs CV 0.289: close, which also confirms our CV setup predicts Kaggle's scoring well.
- Recorded in the `public_lb` column of `experiments.csv`.

## Summary

**Clean data ready for Phase 3:** `X, y, X_test, test_ids, sample = pp.load_clean()` gives 15,167 de-duplicated rows with every airline recovered. `pp.make_preprocessor()` is the per-fold pipeline: duration imputation (86% exact, 0.34 h mean error) plus encoding into 40 numeric columns.

**Usage pattern for every Phase 3 model:**
```python
model = make_pipeline(pp.make_preprocessor(), Ridge(alpha=1.0))
evaluate.cross_validate(model, X, y, "regression", "ridge", params={"alpha": 1.0})
evaluate.cross_validate(evaluate.log_target(model), X, y, "regression", "ridge_log", params={"alpha": 1.0})
```

**Score to beat:** class-median baseline, RMSE ₹33,152 / MAE ₹13,397 / R² 0.29 / RMSLE 0.50.

**Preprocessing points for the report (2 marks):**
1. Exact airline recovery from the flight-code prefix (no guessing).
2. Duplicates removed *after* recovery (279 → 422 found), which prevents CV leakage.
3. Duration imputed from same-flight groups, validated by a masking test (0.34 h vs 2.25 h error for a naive fill).
4. Every data-dependent step re-fitted inside CV folds; rare and unseen airlines handled by an "infrequent" bucket.

**Metric (identified from the baseline submission):** R² on raw price. Maximising R² is the same as minimising squared error in rupees, so the expensive Business tickets matter most. Training on log(price) is no longer the default: it has to beat raw-price training in CV.